# Apache Spark + Apache Iceberg 🧊

**Trabalho de Pesquisa — Engenharia de Dados (SATC)**
Autores: **Murilo Mandelli** e **Augusto Benedetti**

Este notebook demonstra, com PySpark, a criação de tabelas no formato **Apache Iceberg** e os
comandos **INSERT**, **UPDATE** e **DELETE** (além de MERGE, *snapshots* e *time travel*) sobre o
mesmo cenário de seguradora usado no notebook do Delta Lake.

**Roteiro**
1. Cenário e modelo ER
2. Ambiente (SparkSession com catálogo Iceberg)
3. DDL — criação das tabelas (com *hidden partitioning*)
4. Carga inicial a partir dos CSV
5. INSERT
6. UPDATE
7. DELETE
8. MERGE (upsert)
9. Snapshots, *time travel* e rollback
10. Por dentro do Iceberg: metadados e arquivos
11. Consulta analítica final


## 1. Cenário

Uma **seguradora de veículos** precisa manter o cadastro de clientes, carros, apólices e sinistros
em um *lakehouse* local. Usamos o mesmo conjunto de dados do **laboratório de Modelagem
Dimensional** da disciplina (`script_relacional.sql`, banco SQL Server), convertido para CSV pelo
script `scripts/converter_sql_para_csv.py`.

![Modelo ER](../assets/modelo_er.svg)

| Tabela | Linhas | Descrição | Chave |
|---|---:|---|---|
| `cliente` | 20.010 | Pessoas que contratam seguros | `cd_cliente` |
| `carro` | 10.001 | Veículos segurados | `placa` |
| `apolice` | 10.000 | Contrato de seguro: liga **1 cliente** a **1 carro**, com vigência, cobertura e franquia | `cd_apolice` |
| `sinistro` | 10.000 | Ocorrências (acidentes/roubos) envolvendo um carro | `cd_sinistro` |

**Relacionamentos:** `cliente 1 ── N apolice N ── 1 carro 1 ── N sinistro`.

> O Spark **não impõe** chaves primárias/estrangeiras: elas são *lógicas* e servem para os JOINs.


## 2. Ambiente

A função `criar_sessao_iceberg()` (em `src/spark_delta_iceberg/spark_session.py`) cria a
`SparkSession` com:

- `spark.jars.packages = org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.0` → baixa o runtime do Iceberg (Spark 3.5 / Scala 2.12)
- `spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions` → habilita `UPDATE`, `DELETE`, `MERGE` e `CALL`
- `spark.sql.catalog.iceberg = org.apache.iceberg.spark.SparkCatalog` com `type = hadoop` → um **catálogo** chamado `iceberg` que guarda tudo em `data/lakehouse/iceberg/`

No Iceberg as tabelas são referenciadas como **`catalogo.namespace.tabela`** → `iceberg.seguradora.cliente`.

In [1]:
import shutil

from spark_delta_iceberg import TABELAS, criar_sessao_iceberg, ler_csv
from spark_delta_iceberg.config import PACOTE_ICEBERG, PASTA_ICEBERG

shutil.rmtree(PASTA_ICEBERG, ignore_errors=True)

spark = criar_sessao_iceberg()
print("Apache Spark   :", spark.version)
print("Iceberg runtime:", PACOTE_ICEBERG)
print("Warehouse      :", PASTA_ICEBERG)

26/10/05 21:15:19 WARN Utils: Your hostname, DESKTOP-OM56K2B resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/05 21:15:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/root/spark-delta-iceberg/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9fcbd2e7-d519-4dab-904e-0a4971bdae5e;1.0
	confs: [default]


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 in central
:: resolution report :: resolve 325ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-9fcbd2e7-d519-4dab-904e-0a4971bdae5e
	confs: [default]
	0 artifacts copied, 1 already retrieved (0kB/12ms)


26/10/05 21:15:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Apache Spark   : 3.5.9
Iceberg runtime: org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.0
Warehouse      : /root/spark-delta-iceberg/data/lakehouse/iceberg


## 3. DDL — criação das tabelas

Diferenças em relação ao Delta:

- `CREATE NAMESPACE` no lugar de `CREATE DATABASE` (são equivalentes no Spark).
- `USING iceberg`.
- **Hidden partitioning**: `sinistro` é particionada por `years(dt_sinistro)`. O Iceberg calcula o
  ano sozinho — quem consulta não precisa saber da partição nem criar coluna extra.
- Na `apolice` ligamos o modo **merge-on-read** (`write.*.mode`): UPDATE/DELETE gravam pequenos
  *delete files* em vez de reescrever os Parquet inteiros (o padrão é **copy-on-write**).

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.seguradora")

spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.seguradora.cliente (
    cd_cliente     INT     NOT NULL COMMENT 'PK - código do cliente',
    nome           STRING           COMMENT 'Nome completo',
    cpf            STRING  NOT NULL COMMENT 'CPF (texto para manter zeros à esquerda)',
    sexo           STRING           COMMENT 'M/F',
    dt_nascimento  DATE             COMMENT 'Data de nascimento'
) USING iceberg
COMMENT 'Clientes da seguradora'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.seguradora.carro (
    placa      STRING NOT NULL COMMENT 'PK - placa do veículo',
    cd_modelo  INT             COMMENT 'Código do modelo',
    chassi     STRING NOT NULL COMMENT 'Número do chassi',
    ano        INT             COMMENT 'Ano de fabricação',
    cor        STRING          COMMENT 'Cor do veículo'
) USING iceberg
COMMENT 'Veículos segurados'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.seguradora.apolice (
    cd_apolice          INT           NOT NULL COMMENT 'PK - número da apólice',
    cd_cliente          INT           NOT NULL COMMENT 'FK -> cliente',
    dt_inicio_vigencia  TIMESTAMP              COMMENT 'Início da vigência',
    dt_fim_vigencia     TIMESTAMP              COMMENT 'Fim da vigência',
    vl_cobertura        DECIMAL(10,2)          COMMENT 'Valor de cobertura (R$)',
    vl_franquia         DECIMAL(10,2)          COMMENT 'Valor da franquia (R$)',
    placa               STRING                 COMMENT 'FK -> carro'
) USING iceberg
COMMENT 'Apólices: ligam 1 cliente a 1 carro'
TBLPROPERTIES (
    'format-version'    = '2',
    'write.delete.mode' = 'merge-on-read',
    'write.update.mode' = 'merge-on-read',
    'write.merge.mode'  = 'merge-on-read'
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.seguradora.sinistro (
    cd_sinistro     INT       NOT NULL COMMENT 'PK - código do sinistro',
    placa           STRING    NOT NULL COMMENT 'FK -> carro',
    dt_sinistro     TIMESTAMP          COMMENT 'Data/hora da ocorrência',
    local_sinistro  INT                COMMENT 'Código IBGE do município',
    condutor        STRING             COMMENT 'Nome do condutor'
) USING iceberg
PARTITIONED BY (years(dt_sinistro))
COMMENT 'Sinistros (ocorrências) por veículo'
""")

spark.sql("SHOW TABLES IN iceberg.seguradora").show()

+----------+---------+-----------+
| namespace|tableName|isTemporary|
+----------+---------+-----------+
|seguradora|    carro|      false|
|seguradora|  cliente|      false|
|seguradora| sinistro|      false|
|seguradora|  apolice|      false|
+----------+---------+-----------+



In [3]:
# DDL completo gerado pelo Iceberg (repare na partição e nas propriedades)
print(spark.sql("SHOW CREATE TABLE iceberg.seguradora.sinistro").first()[0])

CREATE TABLE iceberg.seguradora.sinistro (
  cd_sinistro INT NOT NULL COMMENT 'PK - código do sinistro',
  placa STRING NOT NULL COMMENT 'FK -> carro',
  dt_sinistro TIMESTAMP COMMENT 'Data/hora da ocorrência',
  local_sinistro INT COMMENT 'Código IBGE do município',
  condutor STRING COMMENT 'Nome do condutor')
USING iceberg
PARTITIONED BY (years(dt_sinistro))
COMMENT 'Sinistros (ocorrências) por veículo'
LOCATION '/root/spark-delta-iceberg/data/lakehouse/iceberg/seguradora/sinistro'
TBLPROPERTIES (
  'current-snapshot-id' = 'none',
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd')



## 4. Carga inicial

Lemos cada CSV com **schema explícito** (`src/spark_delta_iceberg/dados.py`) e gravamos com a API
de DataFrame `writeTo(...).append()` — um **INSERT em lote**. Cada escrita gera um novo
**snapshot** da tabela.

In [4]:
for tabela in TABELAS:
    ler_csv(spark, tabela).writeTo(f"iceberg.seguradora.{tabela}").append()
    print(f"{tabela:<9} {spark.table(f'iceberg.seguradora.{tabela}').count():>6} linhas")

cliente    20010 linhas


carro      10001 linhas


apolice    10000 linhas


sinistro   10000 linhas


In [5]:
spark.table("iceberg.seguradora.apolice").show(5)

+----------+----------+-------------------+-------------------+------------+-----------+-------+
|cd_apolice|cd_cliente| dt_inicio_vigencia|    dt_fim_vigencia|vl_cobertura|vl_franquia|  placa|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
|         1|      4795|2025-02-01 15:25:41|2025-09-09 15:25:41|   554382.00|   77298.00|HEG0631|
|         2|      6043|2023-09-23 06:57:10|2024-06-13 06:57:10|   453879.00|   16425.00|ILM4667|
|         3|     11754|2023-12-03 05:46:51|2024-03-25 05:46:51|   991737.00|   70757.00|QNR4589|
|         4|      1521|2024-02-08 11:45:33|2024-05-14 11:45:33|   432802.00|   87636.00|LWE5122|
|         5|     17956|2023-07-21 06:41:54|2023-10-08 06:41:54|   100988.00|   76181.00|ZDF8957|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
only showing top 5 rows



## 5. INSERT

`INSERT INTO ... VALUES` acrescenta linhas e cria um snapshot do tipo `append`. A gravação é
**atômica**: o novo snapshot só passa a valer quando o arquivo de metadados é trocado.

Cenário: dois novos clientes contratam seguro; um deles já registra um sinistro em Criciúma
(código IBGE `4204608`).

In [6]:
spark.sql("""
INSERT INTO iceberg.seguradora.cliente VALUES
    (20011, 'ANA PAULA SOUZA',     '12345678901', 'F', DATE'1995-04-12'),
    (20012, 'CARLOS EDUARDO LIMA', '10987654321', 'M', DATE'1988-11-30')
""")

spark.sql("INSERT INTO iceberg.seguradora.carro VALUES ('SAT2026', 10, '9BWZZZ377VT004251', 2024, 'PRATA')")

spark.sql("""
INSERT INTO iceberg.seguradora.apolice VALUES
    (10001, 20011, TIMESTAMP'2026-10-01 00:00:00', TIMESTAMP'2027-10-01 00:00:00',
     85000.00, 4500.00, 'SAT2026')
""")

spark.sql("""
INSERT INTO iceberg.seguradora.sinistro VALUES
    (10001, 'SAT2026', TIMESTAMP'2026-10-03 14:30:00', 4204608, 'ANA PAULA')
""")

spark.sql("SELECT * FROM iceberg.seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)
spark.sql("SELECT * FROM iceberg.seguradora.apolice WHERE cd_apolice = 10001").show()

+----------+-------------------+-----------+----+-------------+
|cd_cliente|nome               |cpf        |sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|20011     |ANA PAULA SOUZA    |12345678901|F   |1995-04-12   |
|20012     |CARLOS EDUARDO LIMA|10987654321|M   |1988-11-30   |
+----------+-------------------+-----------+----+-------------+

+----------+----------+-------------------+-------------------+------------+-----------+-------+
|cd_apolice|cd_cliente| dt_inicio_vigencia|    dt_fim_vigencia|vl_cobertura|vl_franquia|  placa|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
|     10001|     20011|2026-10-01 00:00:00|2027-10-01 00:00:00|    85000.00|    4500.00|SAT2026|
+----------+----------+-------------------+-------------------+------------+-----------+-------+



## 6. UPDATE

**Exemplo 1** — o mesmo caso do laboratório: o carro `CXK4218` foi pintado de **PRETO**.
A tabela `carro` usa o modo padrão **copy-on-write**: o arquivo Parquet que contém a linha é reescrito.

In [7]:
print("ANTES")
spark.sql("SELECT placa, ano, cor FROM iceberg.seguradora.carro WHERE placa = 'CXK4218'").show()

spark.sql("UPDATE iceberg.seguradora.carro SET cor = 'PRETO' WHERE placa = 'CXK4218'")

print("DEPOIS")
spark.sql("SELECT placa, ano, cor FROM iceberg.seguradora.carro WHERE placa = 'CXK4218'").show()

ANTES


+-------+----+------+
|  placa| ano|   cor|
+-------+----+------+
|CXK4218|2008|CHUMBO|
+-------+----+------+



DEPOIS
+-------+----+-----+
|  placa| ano|  cor|
+-------+----+-----+
|CXK4218|2008|PRETO|
+-------+----+-----+



**Exemplo 2 (várias linhas)** — reajuste de **10%** na cobertura das apólices iniciadas em 2025.
Como `apolice` está em **merge-on-read**, o Iceberg grava *delete files* + novos dados em vez de
reescrever tudo.

In [8]:
consulta = """
SELECT COUNT(*) AS qtd_apolices, SUM(vl_cobertura) AS total_cobertura
FROM iceberg.seguradora.apolice WHERE YEAR(dt_inicio_vigencia) = 2025
"""
print("ANTES");  spark.sql(consulta).show()

spark.sql("""
UPDATE iceberg.seguradora.apolice
SET vl_cobertura = CAST(vl_cobertura * 1.10 AS DECIMAL(10,2))
WHERE YEAR(dt_inicio_vigencia) = 2025
""")

print("DEPOIS"); spark.sql(consulta).show()

ANTES


+------------+---------------+
|qtd_apolices|total_cobertura|
+------------+---------------+
|         982|   550415493.00|
+------------+---------------+



DEPOIS


+------------+---------------+
|qtd_apolices|total_cobertura|
+------------+---------------+
|         982|   605457042.30|
+------------+---------------+



Arquivos que compõem a tabela agora (`content = 0` → dados, `content = 1` → *position delete
files*). O arquivo original de 10.000 linhas **não foi reescrito**: o Iceberg gravou as 982 linhas
novas e um *delete file* marcando as 982 posições antigas como apagadas.

In [9]:
spark.sql("""
SELECT content, file_format, record_count, file_size_in_bytes
FROM iceberg.seguradora.apolice.files
ORDER BY content, record_count DESC
""").show(truncate=False)

+-------+-----------+------------+------------------+
|content|file_format|record_count|file_size_in_bytes|
+-------+-----------+------------+------------------+
|0      |PARQUET    |10000       |301136            |
|0      |PARQUET    |982         |31851             |
|0      |PARQUET    |1           |2306              |
|1      |PARQUET    |982         |2810              |
+-------+-----------+------------+------------------+



**Exemplo 3** — correção do nome de um cliente:

In [10]:
spark.sql("UPDATE iceberg.seguradora.cliente SET nome = 'CARLOS EDUARDO LIMA FILHO' WHERE cd_cliente = 20012")
spark.sql("SELECT * FROM iceberg.seguradora.cliente WHERE cd_cliente = 20012").show(truncate=False)

+----------+-------------------------+-----------+----+-------------+
|cd_cliente|nome                     |cpf        |sexo|dt_nascimento|
+----------+-------------------------+-----------+----+-------------+
|20012     |CARLOS EDUARDO LIMA FILHO|10987654321|M   |1988-11-30   |
+----------+-------------------------+-----------+----+-------------+



## 7. DELETE

**Exemplo 1** — um sinistro foi lançado por engano. Como `sinistro` é particionada por ano, o
Iceberg só precisa mexer nos arquivos da partição **2026**.

In [11]:
spark.sql("DELETE FROM iceberg.seguradora.sinistro WHERE cd_sinistro = 10001")
spark.sql("SELECT COUNT(*) AS qtd FROM iceberg.seguradora.sinistro WHERE cd_sinistro = 10001").show()

+---+
|qtd|
+---+
|  0|
+---+



**Exemplo 2 (várias linhas)** — expurgo das apólices encerradas antes de julho/2023.
Guardamos o snapshot atual para fazer *rollback* depois.

In [12]:
snapshot_antes_delete = spark.sql(
    "SELECT snapshot_id FROM iceberg.seguradora.apolice.snapshots ORDER BY committed_at DESC LIMIT 1"
).first()["snapshot_id"]

print("Total ANTES :", spark.table("iceberg.seguradora.apolice").count())
spark.sql("DELETE FROM iceberg.seguradora.apolice WHERE dt_fim_vigencia < TIMESTAMP'2023-07-01 00:00:00'")
print("Total DEPOIS:", spark.table("iceberg.seguradora.apolice").count())

Total ANTES : 10001


Total DEPOIS: 9800


**Exemplo 3** — o cliente 20012 desistiu antes de contratar.

In [13]:
spark.sql("DELETE FROM iceberg.seguradora.cliente WHERE cd_cliente = 20012")
spark.sql("SELECT * FROM iceberg.seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)

+----------+---------------+-----------+----+-------------+
|cd_cliente|nome           |cpf        |sexo|dt_nascimento|
+----------+---------------+-----------+----+-------------+
|20011     |ANA PAULA SOUZA|12345678901|F   |1995-04-12   |
+----------+---------------+-----------+----+-------------+



## 8. MERGE (upsert)

`MERGE INTO` atualiza quem já existe e insere quem é novo, em um único snapshot. É o mesmo padrão
usado no laboratório para a carga **SCD Tipo 1** da `dim_cliente`.

In [14]:
atualizacoes = spark.createDataFrame(
    [
        (20011, "ANA PAULA SOUZA MARTINS", "12345678901", "F", "1995-04-12"),  # já existe -> UPDATE
        (20013, "BRUNO HENRIQUE COSTA",    "11122233344", "M", "2001-02-20"),  # novo      -> INSERT
    ],
    "cd_cliente INT, nome STRING, cpf STRING, sexo STRING, dt_nascimento STRING",
).selectExpr("cd_cliente", "nome", "cpf", "sexo", "CAST(dt_nascimento AS DATE) AS dt_nascimento")
atualizacoes.createOrReplaceTempView("cliente_atualizacoes")

spark.sql("""
MERGE INTO iceberg.seguradora.cliente AS destino
USING cliente_atualizacoes AS origem
ON destino.cd_cliente = origem.cd_cliente
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM iceberg.seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)

+----------+-----------------------+-----------+----+-------------+
|cd_cliente|nome                   |cpf        |sexo|dt_nascimento|
+----------+-----------------------+-----------+----+-------------+
|20011     |ANA PAULA SOUZA MARTINS|12345678901|F   |1995-04-12   |
|20013     |BRUNO HENRIQUE COSTA   |11122233344|M   |2001-02-20   |
+----------+-----------------------+-----------+----+-------------+



## 9. Snapshots, *time travel* e rollback

O Iceberg expõe **tabelas de metadados** consultáveis com SQL: `.snapshots`, `.history`, `.files`,
`.partitions`, `.manifests`...

Cada operação gerou um snapshot na tabela `carro`:

In [15]:
spark.sql("""
SELECT committed_at, snapshot_id, parent_id, operation,
       summary['added-records'] AS linhas_add, summary['deleted-records'] AS linhas_del
FROM iceberg.seguradora.carro.snapshots
ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+----------+----------+
|committed_at           |snapshot_id        |parent_id          |operation|linhas_add|linhas_del|
+-----------------------+-------------------+-------------------+---------+----------+----------+
|2026-10-05 21:15:33.211|4394266455667682387|NULL               |append   |10001     |NULL      |
|2026-10-05 21:15:36.385|2643032045395446224|4394266455667682387|append   |1         |NULL      |
|2026-10-05 21:15:38.396|8732939707927455274|2643032045395446224|overwrite|10001     |10001     |
+-----------------------+-------------------+-------------------+---------+----------+----------+



Com `VERSION AS OF <snapshot_id>` consultamos a tabela **como ela era** logo após a carga inicial:

In [16]:
primeiro_snapshot = spark.sql(
    "SELECT snapshot_id FROM iceberg.seguradora.carro.snapshots ORDER BY committed_at LIMIT 1"
).first()["snapshot_id"]

spark.sql(f"""
SELECT 'snapshot da carga' AS versao, placa, cor
FROM iceberg.seguradora.carro VERSION AS OF {primeiro_snapshot} WHERE placa = 'CXK4218'
UNION ALL
SELECT 'versão atual' AS versao, placa, cor
FROM iceberg.seguradora.carro WHERE placa = 'CXK4218'
""").show()

+-----------------+-------+------+
|           versao|  placa|   cor|
+-----------------+-------+------+
|snapshot da carga|CXK4218|CHUMBO|
|     versão atual|CXK4218| PRETO|
+-----------------+-------+------+



A *procedure* `rollback_to_snapshot` volta a tabela para um snapshot anterior — aqui desfazemos
o expurgo de apólices:

In [17]:
print("Antes do rollback:", spark.table("iceberg.seguradora.apolice").count())
spark.sql(
    f"CALL iceberg.system.rollback_to_snapshot('seguradora.apolice', {snapshot_antes_delete})"
).show()
print("Depois do rollback:", spark.table("iceberg.seguradora.apolice").count())

Antes do rollback: 9800


+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
| 1698356442162824968| 659399091093747861|
+--------------------+-------------------+



Depois do rollback: 10001


## 10. Por dentro do Iceberg: metadados e arquivos

Uma tabela Iceberg tem duas pastas: **`data/`** (Parquet, organizados por partição) e
**`metadata/`** (arquivos `vN.metadata.json` → *manifest lists* → *manifests*).

Veja as partições criadas automaticamente pelo *hidden partitioning* na `sinistro`:

In [18]:
# years() guarda "anos desde 1970" -> somamos 1970 para exibir o ano
spark.sql("""
SELECT partition.dt_sinistro_year + 1970 AS ano, record_count, file_count
FROM iceberg.seguradora.sinistro.partitions
ORDER BY ano
""").show()

pasta_sinistro = PASTA_ICEBERG / "seguradora" / "sinistro"
print("Pastas em data/    :", sorted(p.name for p in (pasta_sinistro / "data").iterdir()))
print("Arquivos metadata/ :", len(list((pasta_sinistro / "metadata").iterdir())))

+----+------------+----------+
| ano|record_count|file_count|
+----+------------+----------+
|2023|        2079|         1|
|2024|        2573|         1|
|2025|        2681|         1|
|2026|        2667|         1|
+----+------------+----------+

Pastas em data/    : ['dt_sinistro_year=2023', 'dt_sinistro_year=2024', 'dt_sinistro_year=2025', 'dt_sinistro_year=2026']
Arquivos metadata/ : 22


## 11. Consulta analítica final

Top 5 clientes com mais sinistros (JOIN entre três tabelas Iceberg):

In [19]:
spark.sql("""
SELECT c.cd_cliente, c.nome, COUNT(s.cd_sinistro) AS qtd_sinistros
FROM iceberg.seguradora.cliente  c
JOIN iceberg.seguradora.apolice  a ON a.cd_cliente = c.cd_cliente
JOIN iceberg.seguradora.sinistro s ON s.placa      = a.placa
GROUP BY c.cd_cliente, c.nome
ORDER BY qtd_sinistros DESC, c.cd_cliente
LIMIT 5
""").show(truncate=False)

+----------+----------------+-------------+
|cd_cliente|nome            |qtd_sinistros|
+----------+----------------+-------------+
|1406      |YURI VIEIRA     |5            |
|3263      |LUCCA MARTINS   |5            |
|8756      |RENAN RAMOS     |5            |
|18658     |CAROLINE DA ROSA|5            |
|185       |MAYSA RODRIGUES |4            |
+----------+----------------+-------------+



In [20]:
spark.stop()

## Conclusão

- Criamos 4 tabelas **Iceberg** com DDL SQL (incluindo *hidden partitioning* e *merge-on-read*) e
  carregamos ~50 mil linhas a partir de CSV.
- **INSERT**, **UPDATE**, **DELETE** e **MERGE** funcionaram via SQL.
- Cada operação gerou um **snapshot**, consultável pelas tabelas de metadados, com **time travel**
  (`VERSION AS OF`) e **rollback** (`CALL system.rollback_to_snapshot`).

Documentação completa: veja o site MkDocs do projeto.